# Modulo 4 MNIST classifier

TASK: Classify digit images into four classes - modulo 4 of the corresponding number which can be 0, 1, 2 or 3
- This file is on ILIAS - download it and open in jupyter notebook
- Make sure you understand the code - **data loading**, **preprocessing**, **model** and **loss function** definitions are ready
- **Implement functions _train()_ and _test()_**
- Train the classifier using these functions

## Colab version:
https://colab.research.google.com/drive/1PhAGeXEFfUXUcNftLpVJ5RYBPYKjXOKP  
https://tinyurl.com/yx75zaf6

In [1]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision.datasets import MNIST
from torchvision.transforms import Compose, ToTensor, Normalize

device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
print("You train on {}".format(device))

You train on cpu


In [2]:
# Label transformer function - transforms 0, ... 9 to 0, 1, 2, 3 (mod 4 of the label)
# e.g. label 0 - 0, label 3 - 3, label 5 - 1, label 7 - 3
def label2module4(label):
    return label % 4

# Download the MNIST dataset
Since the last version of Pytorch has a bug and it can't download the MNIST dataset automatically, we need to download and extract to our working folder, first.

In [6]:
!wget www.di.ens.fr/~lelarge/MNIST.tar.gz
!tar -zxvf MNIST.tar.gz

--2021-03-13 03:15:48--  http://www.di.ens.fr/~lelarge/MNIST.tar.gz
Resolving www.di.ens.fr (www.di.ens.fr)... 129.199.99.14
Connecting to www.di.ens.fr (www.di.ens.fr)|129.199.99.14|:80... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://www.di.ens.fr/~lelarge/MNIST.tar.gz [following]
--2021-03-13 03:15:48--  https://www.di.ens.fr/~lelarge/MNIST.tar.gz
Connecting to www.di.ens.fr (www.di.ens.fr)|129.199.99.14|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: unspecified [application/x-gzip]
Saving to: ‘MNIST.tar.gz’

MNIST.tar.gz            [            <=>     ]  33.20M  6.39MB/s    in 15s     

2021-03-13 03:16:03 (2.23 MB/s) - ‘MNIST.tar.gz’ saved [34813078]

MNIST/
MNIST/raw/
MNIST/raw/train-labels-idx1-ubyte
MNIST/raw/t10k-labels-idx1-ubyte.gz
MNIST/raw/t10k-labels-idx1-ubyte
MNIST/raw/t10k-images-idx3-ubyte.gz
MNIST/raw/train-images-idx3-ubyte
MNIST/raw/train-labels-idx1-ubyte.gz
MNIST/raw/t10k-images-idx3-ubyte
MNIST/raw/tra

In [5]:
# Define data transforms: converts Images to tensors and normalizes to [-1, 1]
transforms = Compose([ToTensor(),
                      Normalize(mean=(0.5,), std=(0.5,))])

# Load train dataset
train_dataset = MNIST(root='.', train=True, download=True,
                      transform=transforms,
                      target_transform=label2module4 # transforms labels 0,..,9 to module 4 of these numbers, 0,1,2,3
                     )

# Load test dataset
test_dataset = MNIST(root='.', train=False, download=True,
                      transform=transforms,
                      target_transform=label2module4
                     )

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

In [6]:
class Module4Model(nn.Module):
    def __init__(self):
        super(Module4Model, self).__init__()
        self.main = nn.Sequential(nn.Linear(28*28, 128),
                                 nn.ReLU(),
                                 nn.Linear(128, 4))
    
    def forward(self, x):
        out = x.view(x.size(0), 28*28)
        out = self.main(out)
        return out

In [7]:
model = Module4Model()
model = model.to(device)
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.005)

### Implement train() function
Implement *train()* function, that trains the model for 1 epoch (processes the entire dataset once)
- iterate over train dataset
    - get predictions with **model**
    - compute loss with **loss_fn**, predictions and labels
    - update parameters with **optimizer**
    
You can get inspired by previous tutorials and PyTorch documentation
- https://pytorch.org/docs/stable/optim.html#taking-an-optimization-step
- https://pytorch.org/tutorials/beginner/blitz/cifar10_tutorial.html#train-the-network

In [8]:
def train():
    model.train()
    for iteration, (images, labels) in enumerate(train_loader):
        ## TODO: fill the training loop
        images = images.to(device)
        labels = labels.to(device)
        optimizer.zero_grad()
        output = model(images)
        loss = loss_fn(output, labels)
        loss.backward()
        optimizer.step()
        # The code below will print loss every 100 iterations. Make sure the loss computer by loss_fn is in variable "loss"
        if iteration % 100 == 0:
            print('Training iteration {}: loss {:.4f}'.format(iteration, loss.item()))

### Implement test() function
- iterate over test dataset
    - get predictions with **model**
    - compute loss with **loss_fn**, predictions and labels
    - compute accuracy for the test dataset  
    - assign average test set loss and accuracy to *average_loss* and *accuracy* variables
    
You can get inspired by previous tutorials and PyTorch documentation
https://pytorch.org/tutorials/beginner/blitz/cifar10_tutorial.html#test-the-network-on-the-test-data

In [9]:
def test():
    model.eval()
    # TODO - initialize any variables if necessary
    losses = []
    correct_predictions = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device)
            labels = labels.to(device)
            output = model(images)
            loss = loss_fn(output, labels)
            predicted_labels = output.argmax(dim=1)
            correct_predictions += (predicted_labels == labels).sum().item()
            losses.append(loss.item())

    ## Store and print average test loss and accuracy
    average_loss = sum(losses) / len(losses)
    accuracy = correct_predictions / len(test_dataset)
    print('Test average loss: {:.4f}, accuracy: {:.3f}'.format(average_loss, accuracy))

### Once you've completed train() and test() functions, run the cell below to train and test for 10 epochs
You should be able to get test accuracy > 95 %

In [10]:
n_epochs = 10
for epoch in range(n_epochs):
    print('Epoch {}'.format(epoch+1))
    train()
    test()

Epoch 1
Training iteration 0: loss 1.3682
Training iteration 100: loss 1.1018
Training iteration 200: loss 0.8174
Training iteration 300: loss 0.8307


KeyboardInterrupt: ignored

#### If you succesfully implemented the code, you should have reached around 93%  accuracy on test dataset and a loss of 0.2
#### If you finished early, try to improve the model to increase test accuracy
Things you could try:
- train for more epochs
- play with learning rate value
- increase model size